### Sentinel 2

In [10]:
import ee
import csv
import os
from datetime import datetime
from tqdm import tqdm
from google.colab import files
import concurrent.futures

# --- Danh sách tỉnh ĐBSCL ---
MEKONG_PROVINCES = [
    'An Giang', 'Ben Tre', 'Ca Mau', 'Can Tho city', 'Dong Thap',
    'Hau Giang', 'Kien Giang', 'Long An', 'Soc Trang',
    'Tien Giang', 'Tra Vinh', 'Vinh Long', 'Bac Lieu'
]

# --- Khởi tạo Earth Engine ---
try:
    ee.Initialize(project='ee-python-api-471906')
    print("✅ Earth Engine initialized successfully")
except Exception as e:
    print(f"⚠️ Failed to initialize Earth Engine: {e}")
    ee.Authenticate()
    ee.Initialize(project='ee-python-api-471906')

# --- Hàm lấy vùng ĐBSCL (cache lại) ---
_mekong_region = None
def get_mekong_region():
    global _mekong_region
    if _mekong_region is None:
        provinces = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM0_NAME', 'Viet Nam'))
        mekong_fc = provinces.filter(ee.Filter.inList('ADM1_NAME', MEKONG_PROVINCES))
        _mekong_region = mekong_fc.union().geometry()
    return _mekong_region

# --- Hàm lấy Sentinel-2 collection ---
def get_s2_collection(region, start_date, end_date, cloud_filter=30):
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_filter))
        .sort('CLOUDY_PIXEL_PERCENTAGE')
    )
    size = collection.size().getInfo()
    if size == 0:
        return None, 0
    return collection, size

# --- Hàm batch lấy metadata (quan trọng nhất!) ---
def extract_metadata_batch(collection, total_images):
    """Lấy metadata cho toàn bộ collection chỉ trong 1 lần gọi API"""

    # Lấy tất cả metadata trong 1 lần thay vì từng ảnh
    def get_img_info(img):
        bounds = img.geometry().bounds()
        coords = bounds.coordinates().get(0)

        return ee.Feature(None, {
            'image_id': img.get('system:index'),
            'product_id': img.get('PRODUCT_ID'),
            'time_start': img.get('system:time_start'),
            'coords': coords
        })

    # Map và lấy tất cả info 1 lúc
    features = collection.map(get_img_info).getInfo()['features']

    results = []
    for feat in features:
        props = feat['properties']

        image_id = props.get('image_id', 'NA')
        product_id = props.get('product_id', 'NA')
        millis = props.get('time_start', 0)
        image_date = datetime.fromtimestamp(millis / 1000).strftime('%Y-%m-%d_%H-%M-%S')

        # Bounding box từ coords
        coords = props.get('coords', [[0,0]])
        xs = [c[0] for c in coords]
        ys = [c[1] for c in coords]
        minx, maxx = min(xs), max(xs)
        miny, maxy = min(ys), max(ys)

        results.append([image_id, product_id, image_date, minx, maxx, miny, maxy])

    return results

# --- Hàm xử lý theo năm (parallel) ---
def process_year(year, mekong_region):
    year_results = []

    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        end_date = f"{year+1}-01-01" if month == 12 else f"{year}-{month+1:02d}-01"

        try:
            collection, total_images = get_s2_collection(mekong_region, start_date, end_date, cloud_filter=50)

            if collection is None:
                print(f"⚠️ {month:02d}/{year}: Không có ảnh")
                continue

            monthly_results = extract_metadata_batch(collection, total_images)
            year_results.extend(monthly_results)
            print(f"✅ {month:02d}/{year}: {len(monthly_results)} ảnh")

        except Exception as e:
            print(f"❌ Lỗi {month:02d}/{year}: {e}")
            continue

    return year_results

# --- Hàm chính (tối ưu) ---
def main():
    start_year, end_year = 2019, 2024
    mekong_region = get_mekong_region()
    all_results = []

    print(f"\n🚀 Bắt đầu xử lý {start_year}-{end_year}")
    print("=" * 50)

    # Xử lý tuần tự từng năm (an toàn hơn với GEE quota)
    for year in range(start_year, end_year + 1):
        print(f"\n📅 Năm {year}")
        year_results = process_year(year, mekong_region)
        all_results.extend(year_results)
        print(f"✅ Tổng năm {year}: {len(year_results)} ảnh")

    # --- Lưu toàn bộ kết quả vào CSV ---
    output_csv = f"/content/sentinel-2_{start_year}-{end_year}_bounding_boxes.csv"
    header = ["image_id", "product_id", "image_date", "minx", "maxx", "miny", "maxy"]

    with open(output_csv, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(all_results)

    print(f"\n{'='*50}")
    print(f"📂 Đã lưu {len(all_results)} bounding box vào {output_csv}")

    # --- Nén lại ---
    output_zip = output_csv.replace(".csv", ".zip")
    os.system(f"zip -j {output_zip} {output_csv}")
    print("📦 Đã nén file CSV thành zip.")

    # --- Tải về ---
    files.download(output_zip)

# 🚀 Chạy chính
if __name__ == "__main__":
    main()

✅ Earth Engine initialized successfully

🚀 Bắt đầu xử lý 2019-2024

📅 Năm 2019
✅ 01/2019: 53 ảnh
✅ 02/2019: 75 ảnh
✅ 03/2019: 75 ảnh
✅ 04/2019: 67 ảnh
✅ 05/2019: 56 ảnh
✅ 06/2019: 35 ảnh
✅ 07/2019: 48 ảnh
✅ 08/2019: 26 ảnh
✅ 09/2019: 21 ảnh
✅ 10/2019: 48 ảnh
✅ 11/2019: 56 ảnh
✅ 12/2019: 72 ảnh
✅ Tổng năm 2019: 632 ảnh

📅 Năm 2020
✅ 01/2020: 85 ảnh
✅ 02/2020: 73 ảnh
✅ 03/2020: 82 ảnh
✅ 04/2020: 68 ảnh
✅ 05/2020: 35 ảnh
✅ 06/2020: 16 ảnh
✅ 07/2020: 25 ảnh
✅ 08/2020: 32 ảnh
✅ 09/2020: 23 ảnh
✅ 10/2020: 6 ảnh
✅ 11/2020: 39 ảnh
✅ 12/2020: 43 ảnh
✅ Tổng năm 2020: 527 ảnh

📅 Năm 2021
✅ 01/2021: 45 ảnh
✅ 02/2021: 67 ảnh
✅ 03/2021: 90 ảnh
✅ 04/2021: 48 ảnh
✅ 05/2021: 49 ảnh
✅ 06/2021: 62 ảnh
✅ 07/2021: 25 ảnh
✅ 08/2021: 20 ảnh
✅ 09/2021: 28 ảnh
✅ 10/2021: 15 ảnh
✅ 11/2021: 45 ảnh
✅ 12/2021: 60 ảnh
✅ Tổng năm 2021: 554 ảnh

📅 Năm 2022
✅ 01/2022: 87 ảnh
✅ 02/2022: 45 ảnh
✅ 03/2022: 59 ảnh
✅ 04/2022: 34 ảnh
✅ 05/2022: 21 ảnh
✅ 06/2022: 53 ảnh
✅ 07/2022: 29 ảnh
✅ 08/2022: 26 ảnh
✅ 09/2022: 15 ảnh
✅

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Sentinel 1

In [16]:
import ee
import csv
import os
from datetime import datetime
from google.colab import files

# --- Danh sách tỉnh ĐBSCL ---
MEKONG_PROVINCES = [
    'An Giang', 'Ben Tre', 'Ca Mau', 'Can Tho city', 'Dong Thap',
    'Hau Giang', 'Kien Giang', 'Long An', 'Soc Trang',
    'Tien Giang', 'Tra Vinh', 'Vinh Long', 'Bac Lieu'
]

# --- Khởi tạo Earth Engine ---
try:
    ee.Initialize(project='ee-python-api-471906')
    print("✅ Earth Engine initialized successfully")
except Exception as e:
    print(f"⚠️ Failed to initialize Earth Engine: {e}")
    ee.Authenticate()
    ee.Initialize(project='ee-python-api-471906')

# --- Hàm lấy vùng ĐBSCL (cache lại) ---
_mekong_region = None
def get_mekong_region():
    global _mekong_region
    if _mekong_region is None:
        provinces = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM0_NAME', 'Viet Nam'))
        mekong_fc = provinces.filter(ee.Filter.inList('ADM1_NAME', MEKONG_PROVINCES))
        _mekong_region = mekong_fc.union().geometry()
    return _mekong_region

# --- Hàm lấy Sentinel-1 collection ---
def get_s1_collection(region, start_date, end_date):
    """
    Lấy Sentinel-1 GRD collection
    - IW mode (Interferometric Wide swath)
    - VV+VH polarization
    """
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('resolution_meters', 10))
    )
    size = collection.size().getInfo()
    if size == 0:
        return None, 0
    return collection, size

# --- Hàm batch lấy metadata (tối ưu!) ---
def extract_metadata_batch(collection, total_images):
    """Lấy metadata cho toàn bộ collection chỉ trong 1 lần gọi API"""

    def get_img_info(img):
        bounds = img.geometry().bounds()
        coords = bounds.coordinates().get(0)

        return ee.Feature(None, {
            'image_id': img.get('system:index'),
            'time_start': img.get('system:time_start'),
            'coords': coords
        })

    # Map và lấy tất cả info 1 lúc
    features = collection.map(get_img_info).getInfo()['features']

    results = []
    for feat in features:
        props = feat['properties']

        image_id = props.get('image_id', 'NA')
        millis = props.get('time_start', 0)
        image_date = datetime.fromtimestamp(millis / 1000).strftime('%Y-%m-%d %H:%M:%S')

        # Bounding box từ coords
        coords = props.get('coords', [[0,0]])
        xs = [c[0] for c in coords]
        ys = [c[1] for c in coords]
        minx, maxx = min(xs), max(xs)
        miny, maxy = min(ys), max(ys)

        results.append([image_id, image_date, minx, maxx, miny, maxy])

    return results

# --- Hàm xử lý theo năm ---
def process_year(year, mekong_region):
    year_results = []

    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        end_date = f"{year+1}-01-01" if month == 12 else f"{year}-{month+1:02d}-01"

        try:
            collection, total_images = get_s1_collection(mekong_region, start_date, end_date)

            if collection is None:
                print(f"⚠️ {month:02d}/{year}: Không có ảnh")
                continue

            monthly_results = extract_metadata_batch(collection, total_images)
            year_results.extend(monthly_results)
            print(f"✅ {month:02d}/{year}: {len(monthly_results)} ảnh")

        except Exception as e:
            print(f"❌ Lỗi {month:02d}/{year}: {e}")
            continue

    return year_results

# --- Hàm chính ---
def main():
    start_year, end_year = 2019, 2024
    mekong_region = get_mekong_region()
    all_results = []

    print(f"\n🚀 Bắt đầu xử lý Sentinel-1 {start_year}-{end_year}")
    print("=" * 50)

    # Xử lý tuần tự từng năm
    for year in range(start_year, end_year + 1):
        print(f"\n📅 Năm {year}")
        year_results = process_year(year, mekong_region)
        all_results.extend(year_results)
        print(f"✅ Tổng năm {year}: {len(year_results)} ảnh")

    # --- Lưu toàn bộ kết quả vào CSV ---
    output_csv = f"/content/sentinel-1_{start_year}-{end_year}_bounding_boxes.csv"
    header = ["image_id", "datetime", "minx", "maxx", "miny", "maxy"]

    with open(output_csv, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(all_results)

    print(f"\n{'='*50}")
    print(f"📂 Đã lưu {len(all_results)} bounding box vào {output_csv}")

    # --- Nén lại ---
    output_zip = output_csv.replace(".csv", ".zip")
    os.system(f"zip -j {output_zip} {output_csv}")
    print("📦 Đã nén file CSV thành zip.")

    # --- Tải về ---
    files.download(output_zip)

# 🚀 Chạy chính
if __name__ == "__main__":
    main()

✅ Earth Engine initialized successfully

🚀 Bắt đầu xử lý Sentinel-1 2019-2024

📅 Năm 2019
✅ 01/2019: 42 ảnh
✅ 02/2019: 33 ảnh
✅ 03/2019: 36 ảnh
✅ 04/2019: 39 ảnh
✅ 05/2019: 42 ảnh
✅ 06/2019: 36 ảnh
✅ 07/2019: 37 ảnh
✅ 08/2019: 36 ảnh
✅ 09/2019: 40 ảnh
✅ 10/2019: 35 ảnh
✅ 11/2019: 36 ảnh
✅ 12/2019: 33 ảnh
✅ Tổng năm 2019: 445 ảnh

📅 Năm 2020
✅ 01/2020: 34 ảnh
✅ 02/2020: 33 ảnh
✅ 03/2020: 37 ảnh
✅ 04/2020: 34 ảnh
✅ 05/2020: 39 ảnh
✅ 06/2020: 34 ảnh
✅ 07/2020: 36 ảnh
✅ 08/2020: 37 ảnh
✅ 09/2020: 33 ảnh
✅ 10/2020: 40 ảnh
✅ 11/2020: 38 ảnh
✅ 12/2020: 37 ảnh
✅ Tổng năm 2020: 432 ảnh

📅 Năm 2021
✅ 01/2021: 33 ảnh
✅ 02/2021: 36 ảnh
✅ 03/2021: 27 ảnh
✅ 04/2021: 37 ảnh
✅ 05/2021: 33 ảnh
✅ 06/2021: 36 ảnh
✅ 07/2021: 39 ảnh
✅ 08/2021: 36 ảnh
✅ 09/2021: 35 ảnh
✅ 10/2021: 34 ảnh
✅ 11/2021: 30 ảnh
✅ 12/2021: 32 ảnh
✅ Tổng năm 2021: 408 ảnh

📅 Năm 2022
✅ 01/2022: 28 ảnh
✅ 02/2022: 26 ảnh
✅ 03/2022: 28 ảnh
✅ 04/2022: 27 ảnh
✅ 05/2022: 23 ảnh
✅ 06/2022: 25 ảnh
✅ 07/2022: 23 ảnh
✅ 08/2022: 27 ảnh
✅ 09/20

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 3. Các tỉnh ĐBSCL

In [18]:
import ee
import csv
import os
from google.colab import files

# --- Danh sách tỉnh ĐBSCL ---
MEKONG_PROVINCES = [
    'An Giang', 'Ben Tre', 'Ca Mau', 'Can Tho city', 'Dong Thap',
    'Hau Giang', 'Kien Giang', 'Long An', 'Soc Trang',
    'Tien Giang', 'Tra Vinh', 'Vinh Long', 'Bac Lieu'
]

# --- Khởi tạo Earth Engine ---
try:
    ee.Initialize(project='ee-python-api-471906')
    print("✅ Earth Engine initialized successfully")
except Exception as e:
    print(f"⚠️ Failed to initialize Earth Engine: {e}")
    ee.Authenticate()
    ee.Initialize(project='ee-python-api-471906')

# --- Hàm lấy bounding box của các tỉnh ĐBSCL ---
def get_provinces_bbox():
    provinces = ee.FeatureCollection("FAO/GAUL/2015/level1") \
        .filter(ee.Filter.eq('ADM0_NAME', 'Viet Nam')) \
        .filter(ee.Filter.inList('ADM1_NAME', MEKONG_PROVINCES))

    results = []
    for p in provinces.getInfo()['features']:
        name = p['properties']['ADM1_NAME']

        # Lấy geometry hợp nhất và bounding box
        geom = ee.Geometry(p['geometry'])
        bounds = geom.bounds().getInfo()['coordinates'][0]

        xs = [c[0] for c in bounds]
        ys = [c[1] for c in bounds]
        minx, maxx = min(xs), max(xs)
        miny, maxy = min(ys), max(ys)

        results.append([name, minx, maxx, miny, maxy])
    return results

# --- Xuất ra CSV ---
def export_bbox_csv():
    province_results = get_provinces_bbox()

    output_csv = "/content/mekong_provinces_bbox.csv"
    header = ["province", "minx", "maxx", "miny", "maxy"]

    with open(output_csv, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(province_results)

    print(f"📂 Đã lưu bounding box 13 tỉnh ĐBSCL vào {output_csv}")

    # Nén lại
    output_zip = output_csv.replace(".csv", ".zip")
    os.system(f"zip -j {output_zip} {output_csv}")
    print("📦 Đã nén file CSV thành zip.")

    # Tải về
    files.download(output_zip)

# 🚀 Chạy chính
export_bbox_csv()


✅ Earth Engine initialized successfully
📂 Đã lưu bounding box 13 tỉnh ĐBSCL vào /content/mekong_provinces_bbox.csv
📦 Đã nén file CSV thành zip.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>